# 🌦 Weather Forecasting MLOps — Full Pipeline
**Pipeline:** Fetch Data → EDA → Feature Engineering → Train Prophet & LSTM → Evaluate → Champion/Challenger → Dynamic Ensemble

Chạy trên Google Colab với GPU Runtime.

In [ ]:
!pip install -q prophet torch pandas numpy scikit-learn matplotlib requests

## 1. Fetch Data từ Open-Meteo API
Lấy dữ liệu thời tiết lịch sử 1 năm cho 3 thành phố: Hà Nội, Đà Nẵng, HCM.

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

CITIES = {
    'hanoi': {'lat': 21.0285, 'lon': 105.8542, 'name': 'Hà Nội'},
    'danang': {'lat': 16.0544, 'lon': 108.2022, 'name': 'Đà Nẵng'},
    'hcm': {'lat': 10.8231, 'lon': 106.6297, 'name': 'TP.HCM'},
}

def fetch_city_data(city_id, days=365):
    c = CITIES[city_id]
    params = {
        'latitude': c['lat'], 'longitude': c['lon'],
        'start_date': (pd.Timestamp.now() - pd.Timedelta(days=days)).strftime('%Y-%m-%d'),
        'end_date': (pd.Timestamp.now() - pd.Timedelta(days=1)).strftime('%Y-%m-%d'),
        'hourly': 'temperature_2m,relative_humidity_2m,surface_pressure,dew_point_2m,precipitation,wind_speed_10m,cloud_cover',
        'timezone': 'Asia/Bangkok'
    }
    resp = requests.get('https://archive-api.open-meteo.com/v1/archive', params=params)
    df = pd.DataFrame(resp.json()['hourly'])
    df['ds'] = pd.to_datetime(df['time'])
    df['city_id'] = city_id
    df = df.rename(columns={'temperature_2m':'y','surface_pressure':'pressure','dew_point_2m':'dewpoint',
                            'relative_humidity_2m':'humidity','wind_speed_10m':'wind_speed'})
    return df.dropna()

print('Fetching data...')
all_data = {}
for cid in CITIES:
    all_data[cid] = fetch_city_data(cid)
    print(f'  {cid}: {len(all_data[cid])} rows')

df_all = pd.concat(all_data.values(), ignore_index=True)
print(f'\nTotal: {len(df_all)} rows')
df_all.head()

## 2. Khám phá Dữ liệu (EDA)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Temperature distribution
for cid in CITIES:
    df_c = all_data[cid]
    axes[0,0].plot(df_c['ds'].iloc[::24], df_c['y'].rolling(24).mean().iloc[::24], label=cid, alpha=0.8)
axes[0,0].set_title('Nhiệt độ trung bình ngày'); axes[0,0].legend()

# Humidity
for cid in CITIES:
    all_data[cid]['humidity'].hist(ax=axes[0,1], alpha=0.5, label=cid, bins=30)
axes[0,1].set_title('Phân phối Độ ẩm'); axes[0,1].legend()

# Precipitation
df_hanoi = all_data['hanoi']
daily_rain = df_hanoi.groupby(df_hanoi['ds'].dt.date)['precipitation'].sum()
axes[1,0].bar(range(len(daily_rain)), daily_rain.values, width=1, alpha=0.7)
axes[1,0].set_title('Lượng mưa hàng ngày (Hà Nội)')

# Correlation heatmap
corr_cols = ['y','humidity','pressure','dewpoint','precipitation','wind_speed','cloud_cover']
corr = df_hanoi[corr_cols].corr()
im = axes[1,1].imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
axes[1,1].set_xticks(range(len(corr_cols))); axes[1,1].set_xticklabels(corr_cols, rotation=45, ha='right')
axes[1,1].set_yticks(range(len(corr_cols))); axes[1,1].set_yticklabels(corr_cols)
axes[1,1].set_title('Correlation Matrix')
plt.colorbar(im, ax=axes[1,1])

plt.tight_layout()
plt.show()

## 3. Data Cleaning

In [ ]:
def clean_data(df):
    print(f'Before cleaning: {len(df)} rows')
    print(f'Missing values:\n{df.isnull().sum()}')

    # Remove duplicates
    df = df.drop_duplicates(subset=['ds', 'city_id'])

    # Remove outliers (temperature > 50 or < -10)
    df = df[(df['y'] > -10) & (df['y'] < 50)]

    # Fill small gaps
    df = df.sort_values('ds').ffill().bfill()

    print(f'After cleaning: {len(df)} rows')
    return df

df_clean = clean_data(df_all)

## 4. Feature Engineering (Domain Knowledge Thời tiết)
Thêm 4 features chuyên sâu dựa trên kiến thức khí tượng học.

In [ ]:
def add_weather_features(df, is_hourly=True):
    df = df.copy()

    # Temporal features
    df['day_of_week'] = df['ds'].dt.dayofweek
    df['month'] = df['ds'].dt.month
    if is_hourly:
        df['hour'] = df['ds'].dt.hour

    # Lag features
    lag_list = [1, 2, 3, 6, 12, 24] if is_hourly else [1, 2, 3, 7]
    for lag in lag_list:
        df[f'temp_lag_{lag}'] = df['y'].shift(lag)

    # Domain-specific weather features
    rolling_w = 7*24 if is_hourly else 7
    df['temp_rolling_7d'] = df['y'].rolling(window=rolling_w, min_periods=1).mean()  # Xu hướng trung hạn

    amp_w = 24 if is_hourly else 1
    df['temp_amplitude'] = df['y'].rolling(window=amp_w, min_periods=1).max() - df['y'].rolling(window=amp_w, min_periods=1).min()  # Biên độ nhiệt

    if 'pressure' in df.columns:
        p_lag = 24 if is_hourly else 1
        df['pressure_change_24h'] = df['pressure'] - df['pressure'].shift(p_lag)  # Front lạnh/bão

    if 'dewpoint' in df.columns:
        df['dewpoint_spread'] = df['y'] - df['dewpoint']  # Khả năng mưa

    return df.dropna()

# Apply per city
featured_data = {}
for cid in CITIES:
    df_c = df_clean[df_clean['city_id'] == cid].sort_values('ds').copy()
    featured_data[cid] = add_weather_features(df_c)
    print(f'{cid}: {featured_data[cid].shape}')

print('\nFeatures:', featured_data['hanoi'].columns.tolist())

## 5. Train Prophet Model
Prophet với extra regressors (domain features + lag features).

In [ ]:
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

REGRESSOR_COLS = ['temp_lag_1', 'temp_lag_24', 'temp_rolling_7d', 'temp_amplitude', 'pressure_change_24h', 'dewpoint_spread']
VAL_DAYS = 14  # 14 ngày cuối làm tập validation

prophet_models = {}
prophet_metrics = {}

for cid in CITIES:
    print(f'\n--- Training Prophet for {cid} ---')
    df = featured_data[cid].copy()

    split = len(df) - VAL_DAYS * 24
    train_df = df.iloc[:split]
    val_df = df.iloc[split:]

    model = Prophet(
        changepoint_prior_scale=0.05,
        seasonality_prior_scale=10.0,
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=True,
    )
    for col in REGRESSOR_COLS:
        model.add_regressor(col)

    model.fit(train_df)
    prophet_models[cid] = model

    # Evaluate
    forecast = model.predict(val_df)
    y_true = val_df['y'].values
    y_pred = forecast['yhat'].values
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    prophet_metrics[cid] = {'mae': round(mae, 3), 'rmse': round(rmse, 3)}
    print(f'  MAE: {mae:.3f}°C, RMSE: {rmse:.3f}°C')

print('\n=== Prophet Results ===')
for cid, m in prophet_metrics.items():
    print(f'  {cid}: MAE={m["mae"]}°C, RMSE={m["rmse"]}°C')

## 6. Visualize Prophet Results + Confidence Intervals

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=False)

for i, cid in enumerate(CITIES):
    df = featured_data[cid]
    split = len(df) - VAL_DAYS * 24
    val_df = df.iloc[split:]
    forecast = prophet_models[cid].predict(val_df)

    ax = axes[i]
    ax.plot(val_df['ds'], val_df['y'], label='Actual', color='#2196F3', linewidth=1)
    ax.plot(val_df['ds'], forecast['yhat'], label='Prophet', color='#FF5722', linewidth=1)
    ax.fill_between(val_df['ds'], forecast['yhat_lower'], forecast['yhat_upper'],
                    color='#FF5722', alpha=0.15, label='Confidence Interval')
    ax.set_title(f'{CITIES[cid]["name"]} — MAE: {prophet_metrics[cid]["mae"]}°C')
    ax.legend(loc='upper right')
    ax.set_ylabel('°C')

plt.tight_layout()
plt.show()

## 7. Model Explainability — Prophet Components
Phân tích trend, seasonality (yearly/weekly/daily) của model.

In [ ]:
# Chọn Hanoi để phân tích components
df = featured_data['hanoi']
forecast_full = prophet_models['hanoi'].predict(df)
fig = prophet_models['hanoi'].plot_components(forecast_full)
plt.suptitle('Prophet Components — Hanoi', y=1.02, fontsize=14)
plt.show()

## 8. Train LSTM Model (Multi-city)
LSTM Encoder-Decoder: input 24h × 3 cities → predict 72h × 3 cities.

In [ ]:
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler

# === Constants ===
LOOKBACK = 24
HORIZON = 72
N_CITIES = 3
EPOCHS = 50
BATCH_SIZE = 32
LR = 0.001

# === Prepare multi-city matrix ===
city_ids = sorted(CITIES.keys())
dfs = []
for cid in city_ids:
    s = all_data[cid][['ds', 'y']].copy()
    s = s.rename(columns={'y': cid})
    dfs.append(s.set_index('ds'))

df_multi = pd.concat(dfs, axis=1).ffill().bfill().dropna()
print(f'Multi-city matrix: {df_multi.shape}')

# Normalize
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(df_multi.values)

# Sliding window
def create_sequences(data, lookback, horizon):
    X, y = [], []
    for i in range(len(data) - lookback - horizon + 1):
        X.append(data[i:i+lookback])
        y.append(data[i+lookback:i+lookback+horizon])
    return np.array(X), np.array(y)

X, y = create_sequences(data_scaled, LOOKBACK, HORIZON)
print(f'X: {X.shape}, y: {y.shape}')

# Train/Val split (80/20 chronological)
split = int(len(X) * 0.8)
X_train, X_val = torch.FloatTensor(X[:split]), torch.FloatTensor(X[split:])
y_train, y_val = torch.FloatTensor(y[:split]), torch.FloatTensor(y[split:])
print(f'Train: {X_train.shape}, Val: {X_val.shape}')

In [ ]:
# === LSTM Model ===
class LSTMWeatherModel(nn.Module):
    def __init__(self, n_features=3, hidden=128, horizon=72):
        super().__init__()
        self.encoder = nn.LSTM(n_features, hidden, batch_first=True, dropout=0.2)
        self.decoder_fc = nn.Linear(hidden, 64)
        self.output_fc = nn.Linear(64, n_features * horizon)
        self.n_features = n_features
        self.horizon = horizon

    def forward(self, x):
        _, (h, _) = self.encoder(x)
        h = torch.relu(self.decoder_fc(h.squeeze(0)))
        out = self.output_fc(h)
        return out.view(-1, self.horizon, self.n_features)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model_lstm = LSTMWeatherModel(n_features=N_CITIES, hidden=128, horizon=HORIZON).to(device)
optimizer = torch.optim.Adam(model_lstm.parameters(), lr=LR)
criterion = nn.MSELoss()

# Training loop
train_losses, val_losses = [], []
best_val = float('inf')
patience, patience_counter = 10, 0

for epoch in range(EPOCHS):
    model_lstm.train()
    total_loss = 0
    for i in range(0, len(X_train), BATCH_SIZE):
        xb = X_train[i:i+BATCH_SIZE].to(device)
        yb = y_train[i:i+BATCH_SIZE].to(device)
        pred = model_lstm(xb)
        loss = criterion(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    train_loss = total_loss / (len(X_train) // BATCH_SIZE)
    train_losses.append(train_loss)

    model_lstm.eval()
    with torch.no_grad():
        val_pred = model_lstm(X_val.to(device))
        val_loss = criterion(val_pred, y_val.to(device)).item()
    val_losses.append(val_loss)

    if val_loss < best_val:
        best_val = val_loss
        patience_counter = 0
        best_state = model_lstm.state_dict().copy()
    else:
        patience_counter += 1

    if (epoch+1) % 5 == 0:
        print(f'Epoch {epoch+1}/{EPOCHS} | Train: {train_loss:.6f} | Val: {val_loss:.6f}')

    if patience_counter >= patience:
        print(f'Early stopping at epoch {epoch+1}')
        break

model_lstm.load_state_dict(best_state)
print(f'Best val loss: {best_val:.6f}')

## 9. Training Curves

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('LSTM Training Curve')
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

## 10. Evaluate LSTM

In [ ]:
model_lstm.eval()
with torch.no_grad():
    preds = model_lstm(X_val.to(device)).cpu().numpy()

# Denormalize
y_val_real = scaler.inverse_transform(y_val.reshape(-1, N_CITIES))
preds_real = scaler.inverse_transform(preds.reshape(-1, N_CITIES))

lstm_metrics = {}
for i, cid in enumerate(city_ids):
    mae = mean_absolute_error(y_val_real[:, i], preds_real[:, i])
    rmse = np.sqrt(mean_squared_error(y_val_real[:, i], preds_real[:, i]))
    lstm_metrics[cid] = {'mae': round(mae, 3), 'rmse': round(rmse, 3)}
    print(f'LSTM {cid}: MAE={mae:.3f}°C, RMSE={rmse:.3f}°C')

## 11. So sánh Prophet vs LSTM + Champion/Challenger

In [ ]:
print('=' * 50)
print(f'{"City":<10} {"Prophet MAE":>12} {"LSTM MAE":>12} {"Winner":>10}')
print('=' * 50)

for cid in city_ids:
    p_mae = prophet_metrics[cid]['mae']
    l_mae = lstm_metrics[cid]['mae']
    winner = 'Prophet' if p_mae < l_mae else 'LSTM'
    print(f'{cid:<10} {p_mae:>12.3f} {l_mae:>12.3f} {winner:>10}')

print('\n--- Champion/Challenger Logic ---')
RMSE_TOLERANCE = 0.05
old_mae, old_rmse = 2.0, 2.5  # Giả lập model cũ

for cid in city_ids:
    new_mae = prophet_metrics[cid]['mae']
    new_rmse = prophet_metrics[cid]['rmse']
    if new_mae <= old_mae and new_rmse <= old_rmse * (1 + RMSE_TOLERANCE):
        print(f'  {cid}: ✅ ACCEPT (MAE {new_mae} <= {old_mae})')
    else:
        print(f'  {cid}: ❌ REJECT (MAE {new_mae} vs {old_mae})')

## 12. Dynamic Ensemble Weights

In [ ]:
def dynamic_weights(prophet_mae, lstm_mae):
    inv_p = 1.0 / prophet_mae
    inv_l = 1.0 / lstm_mae
    pw = max(0.3, min(0.7, inv_p / (inv_p + inv_l)))
    return pw, 1.0 - pw

print('Dynamic Ensemble Weights per City:')
for cid in city_ids:
    wp, wl = dynamic_weights(prophet_metrics[cid]['mae'], lstm_metrics[cid]['mae'])
    print(f'  {cid}: Prophet={wp:.0%}, LSTM={wl:.0%}')

## 13. Final Comparison Chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(city_ids))
w = 0.35
bars1 = ax.bar(x - w/2, [prophet_metrics[c]['mae'] for c in city_ids], w, label='Prophet', color='#2196F3')
bars2 = ax.bar(x + w/2, [lstm_metrics[c]['mae'] for c in city_ids], w, label='LSTM', color='#FF5722')

ax.set_ylabel('MAE (°C)'); ax.set_title('Prophet vs LSTM — MAE per City')
ax.set_xticks(x); ax.set_xticklabels([CITIES[c]['name'] for c in city_ids])
ax.legend(); ax.grid(axis='y', alpha=0.3)

for bar in bars1 + bars2:
    h = bar.get_height()
    ax.annotate(f'{h:.2f}', xy=(bar.get_x() + bar.get_width()/2, h), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## ✅ Summary
| Feature | Detail |
|---|---|
| **Data** | Open-Meteo API, 3 cities, 1 year hourly |
| **Feature Eng.** | 6 lag + 4 domain features (rolling 7d, amplitude, pressure change, dewpoint spread) |
| **Prophet** | Per-city, with extra regressors + confidence intervals |
| **LSTM** | Multi-city encoder-decoder (24h→72h) |
| **Ensemble** | Dynamic weights (inverse MAE, clamped [0.3, 0.7]) |
| **Evaluation** | Champion/Challenger: MAE ≤ old + RMSE tolerance 5% |